# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kavithamuddagowni3-cyber/flyrank/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [ ]:
## 1. Unit of analysis + time window

The unit of analysis in this project is:

**One row = one anonymized content page with aggregated performance signals.**

Each row represents a content item with its search and engagement metrics collected over a 90-day observation window.

The starter dataset contains aggregated features such as:
- impressions_90d
- clicks_90d
- sessions_90d
- CTR
- average position
- engagement rate
- content age

The model uses these historical signals to identify pages that may require content refresh review.

The dataset is a snapshot of observed content performance. The starter label (`is_declining_label`) is created from the current trend direction rather than a future outcome, so the result should be treated as decision support rather than causal prediction.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
## 2. Fields: feature / label / context / excluded

The fields used in this project are divided into four categories based on their role in the ML workflow.

### Features (used as model inputs)

These are observable signals available before making a review decision:

- impressions_90d — search visibility over the last 90 days
- clicks_90d — search clicks over the last 90 days
- sessions_90d — user sessions over the last 90 days
- ai_sessions_90d — AI referral sessions where available
- ctr — click-through rate
- avg_position — average search ranking position
- content_age_days — age of the content
- word_count — content length indicator
- engagement_rate — user engagement measurement
- scroll_rate — content interaction signal
- competition level — content/search competition category
- content type — type of content
- intent — content intent category

### Label (what the model predicts)

- `is_declining_label`

This label is created from:

`trend_direction == "down"`

It represents whether a page is currently showing a declining trend. This is a proxy label based on observed data, not a guaranteed future outcome.

### Context fields (used for interpretation, not prediction)

- content_id — anonymized content identifier
- client_id — anonymized client identifier
- age/freshness tiers
- position tiers
- impression tiers
- trend_direction

These fields help understand results and create reason codes.

### Excluded fields and why

- Raw URLs — excluded because they may reveal private information.
- Raw queries/keywords — excluded because they contain sensitive search information.
- Client names/domains — excluded to protect confidentiality.
- Product decision fields such as `health_score`, `priority_score`, or `action_type` — excluded because they are existing rule outputs and using them would create leakage or simply copy existing decisions.

The model is built only using observable signals so that it learns patterns from data rather than reproducing existing product rules.

import pandas as pd

# Load processed features
df = pd.read_csv("../../data/processed/refresh_feature_vector.csv")

# Show all available columns
print("Columns used in dataset:")
for col in df.columns:
    print("-", col)

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
## 3. Verify it with queries (grain, counts, missing values, windows)

The data contract claims are verified using checks on the actual dataframe.

The verification checks include:

- Dataset size to confirm the number of content rows.
- Duplicate checks to confirm one row represents one content item.
- Missing value checks to understand data quality.
- Column checks to verify required features exist.
- Value checks to confirm the target label distribution.

These checks ensure that the dataset structure matches the assumptions used for the ML task.

import pandas as pd

# Load processed feature dataset
df = pd.read_csv("../../data/processed/refresh_feature_vector.csv")

print("===== DATASET SIZE =====")
print("Rows:", len(df))
print("Columns:", len(df.columns))


print("\n===== COLUMN CHECK =====")
required_columns = [
    "content_id",
    "impressions_90d",
    "sessions_90d",
    "ctr",
    "avg_position",
    "content_age_days",
    "is_declining_label"
]

for col in required_columns:
    print(col, ":", col in df.columns)


print("\n===== DUPLICATE CHECK =====")
print("Duplicate content IDs:", df["content_id"].duplicated().sum())


print("\n===== MISSING VALUES =====")
print(df[required_columns].isnull().sum())


print("\n===== LABEL DISTRIBUTION =====")
print(df["is_declining_label"].value_counts())


print("\n===== LABEL PERCENTAGE =====")
print(
    (df["is_declining_label"].value_counts(normalize=True) * 100)
    .round(2)
)

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [ ]:
## 4. Data limits

This dataset provides useful observable signals, but it has important limitations.

### What this data can tell us:
- It can show patterns between search visibility, engagement, content age, and performance changes.
- It can help rank pages that deserve further human review.
- It can support decision-making using historical content performance signals.

### What this data can never tell us:

- It cannot prove that updating a page will cause traffic or ranking improvement because there is no causal experiment.
- It cannot predict Google's algorithm or explain exact ranking factors.
- It cannot guarantee that a high-priority page will recover after a refresh.

### Dataset limitations:

- The history is unbalanced because different content items and clients may have different amounts of available data.
- Some early rows may contain search data only when analytics tracking was not available, so missing engagement data should not automatically mean poor performance.
- The starter dataset uses aggregated 90-day signals, which limits detailed time-series analysis.
- The current label is based on an observed trend rule (`trend_direction == "down"`), not a future outcome.
- Feature windows and target definitions must be carefully separated in future models to avoid leakage from overlapping time periods.

These limitations mean the model should be used as a ranking and decision-support system, not as a guarantee of future content performance.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.